In [2]:
import os
import pickle
import gc
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import xarray as xr
import tensorflow as tf
import shap
import seaborn as sns
import sys
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.models import load_model
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import Polygon, Point

# ===== CONFIGURATION =====
# Path to the ReLU model
model_path = "water_demand_convlstm2d_relu_model.keras"
model_name = "ConvLSTM2D ReLU"

# Path to the WMA shapefile
wma_shapefile_path = "work/yifanl/ConvLSTM/simpleWMAs_v3_ID.shp"
# Assumed ID field in the shapefile for merging (verify this matches your shapefile)
shapefile_id_field = 'ID'

# Dictionary to store feature names
FEATURE_NAMES = {
    0: "Temperature",
    1: "Precipitation",
    2: "Population Density",
    3: "Domestic Water Usage",
    4: "Industrial Water Usage",
    5: "Crop Area Fraction",
    6: "Irrigation Efficiency"
}

# Make sure output directory exists
output_base_dir = "analysis_outputs"
shap_output_dir = os.path.join(output_base_dir, "shap_plots")
wma_output_dir = os.path.join(output_base_dir, "wma_maps")

os.makedirs(output_base_dir, exist_ok=True)
os.makedirs(shap_output_dir, exist_ok=True)
os.makedirs(wma_output_dir, exist_ok=True)
os.makedirs(os.path.join(wma_output_dir, "top_wmas"), exist_ok=True)


def close_figures():
    """Helper function to explicitly close all open matplotlib figures"""
    plt.close('all')

# ===== LOAD MODEL AND DATA =====
print("\n===== Loading Model and Data =====")

# Close any existing figures at the start
close_figures()
# Load model
print(f"Loading {model_name} from {model_path}...")
try:
    model = load_model(model_path)
    print(f"{model_name} loaded successfully.")
except Exception as e:
    print(f"Error loading model from {model_path}: {e}")
    print("Please ensure the model file exists and is compatible with your TensorFlow installation.")
    import traceback
    traceback.print_exc()
    sys.exit(1) # Exit if model cannot be loaded

# Load analysis data
analysis_data_file = os.path.join(output_base_dir, 'water_demand_model_data.pkl')
print(f"Loading analysis data from {analysis_data_file}...")
try:
    with open(analysis_data_file, 'rb') as f:
        analysis_data = pickle.load(f)

    # Extract data components
    X_test = analysis_data.get('X_test')
    y_test = analysis_data.get('y_test')
    # Assume WMA IDs are needed and loaded
    ids_test = analysis_data.get('ids_test')

    if X_test is None or y_test is None:
        raise ValueError("Required data (X_test, y_test) not found in the pickle file.")
    # Ensure ids_test is available for WMA mapping
    if ids_test is None:
         print("Warning: ids_test not found in analysis data. WMA mapping will be skipped.")


    print(f"Data loaded successfully:")
    print(f"  - X_test shape: {X_test.shape}")
    print(f"  - y_test shape: {y_test.shape}")
    if ids_test is not None:
        print(f"  - ids_test shape: {ids_test.shape}")

except FileNotFoundError:
    print(f"Error: Analysis data file not found at {analysis_data_file}")
    print("Please ensure the file exists and run the data preparation step.")
    sys.exit(1) # Exit if data is not found
except Exception as e:
    print(f"Error loading analysis data: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1) # Exit on other data loading errors


# ===== COMPREHENSIVE SHAP ANALYSIS AND WMA MAPPING (COMBINED) =====
def run_analysis(model, X_test, y_test, ids_test=None, max_samples_shap=20000):
    """
    Run comprehensive SHAP analysis and map results to WMAs.

    Parameters:
    -----------
    model : tensorflow.keras.Model
        The model to analyze
    X_test : numpy.ndarray
        Test data features
    y_test : numpy.ndarray
        Test data labels
    ids_test : numpy.ndarray, optional
        Water Management Area IDs for test data (MUST match IDs used in the shapefile's ID_FIELD)
    max_samples_shap : int, default=20000
        Maximum number of samples to use for SHAP analysis calculation
    """
    print("\n===== RUNNING ANALYSIS (SHAP + WMA Mapping) =====")

    # Extract dimensions
    if X_test.ndim != 5:
        print(f"Error: X_test has unexpected dimensions {X_test.ndim}. Expected 5.")
        print("Analysis failed.")
        return None # Return None if input shape is wrong

    n_samples, seq_len, height, width, n_features = X_test.shape

    try:
        # --- SHAP Calculation Section ---
        print("\n--- Calculating SHAP Values ---")
        # Subset data for SHAP calculation if needed
        if ids_test is not None and len(ids_test) != n_samples:
             print("Error: ids_test length does not match X_test length.")
             return None # Exit if IDs and data don't match

        if n_samples > max_samples_shap:
            print(f"Selecting {max_samples_shap} random samples for SHAP analysis...")
            np.random.seed(42)
            indices = np.random.choice(n_samples, max_samples_shap, replace=False)
            X_sample = X_test[indices]
            y_sample = y_test[indices]
            ids_sample = ids_test[indices] if ids_test is not None else None # Subset IDs too!
        else:
            X_sample = X_test
            y_sample = y_test
            ids_sample = ids_test # Use all IDs if using all data

        print(f"SHAP analysis on {len(X_sample)} samples...")

        # Create a simplified version of the data for SHAP by averaging
        X_time_features = np.mean(X_sample, axis=(2, 3)) # Average over spatial dimensions
        X_features_avg = np.mean(X_time_features, axis=1) # Further average over time

        # Clean data for SHAP (handling potential NaNs or infinities)
        X_features_avg_clean = np.nan_to_num(X_features_avg, nan=0.0, posinf=1e10, neginf=-1e10)
        y_sample_clean = np.nan_to_num(y_sample, nan=0.0, posinf=1e10, neginf=-1e10)

        # Create a surrogate model for SHAP to explain (simpler model)
        print("Training surrogate model for SHAP analysis...")
        surrogate_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        surrogate_model.fit(X_features_avg_clean, y_sample_clean)

        # Create SHAP explainer
        print("Creating SHAP explainer...")
        explainer = shap.TreeExplainer(surrogate_model)

        # Calculate SHAP values
        print("Calculating SHAP values...")
        shap_values = explainer.shap_values(X_features_avg_clean)
        expected_value = explainer.expected_value

        # Store feature names for easier reference
        feature_names = [FEATURE_NAMES.get(i, f'Feature {i}') for i in range(n_features)]

        # Calculate feature importance based on mean absolute SHAP values
        mean_abs_shap = np.abs(shap_values).mean(0)
        feature_importance = mean_abs_shap / (np.sum(mean_abs_shap) or 1e-9) # Avoid division by zero
        feature_ranks = np.argsort(feature_importance)[::-1]

        # Save overall feature importance results
        results_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': feature_importance,
            'Mean_Abs_SHAP': mean_abs_shap
        })
        results_df = results_df.sort_values('Importance', ascending=False)
        results_df.to_csv(f"{shap_output_dir}/feature_importance_shap_{model_name.replace(' ', '_').lower()}.csv", index=False)

        print("\nSHAP Feature Importance Results:")
        print(results_df)

        # --- SHAP Plotting (Non-Map) ---
        print("\n--- Generating SHAP Plots ---")
        # (Code for Summary, Dependence, Force, Heatmap, Decision, Waterfall, Beeswarm, Bar plots)
        # This section remains largely the same as in the first script, using shap_values, expected_value, X_features_avg_clean, and feature_names

        # 1. Summary Plot
        print("\nCreating SHAP Summary Plot...")
        plt.figure(figsize=(12, 8))
        shap.summary_plot(
            shap_values,
            X_features_avg_clean,
            feature_names=feature_names,
            show=False
        )
        plt.title(f"SHAP Summary Plot: {model_name}")
        plt.tight_layout()
        plt.savefig(f"{shap_output_dir}/1_summary_plot.png", dpi=300)
        plt.close()


        # 2. Dependence Plot for all features
        print("\nCreating SHAP Dependence Plots for all features...")
        for i, feature in enumerate(feature_names):
             try:
                plt.figure(figsize=(10, 6))
                shap.dependence_plot(
                    i,
                    shap_values,
                    X_features_avg_clean,
                    feature_names=feature_names,
                    show=False,
                    alpha=0.5
                )
                plt.title(f"SHAP Dependence Plot for {feature}")
                plt.tight_layout()
                plt.savefig(f"{shap_output_dir}/2_dependence_plot_{feature.lower().replace(' ', '_')}.png", dpi=300)
                plt.close()
             except Exception as dep_err:
                 print(f"Could not create dependence plot for {feature}: {dep_err}")


        # 3. Force Plot for individual samples
        print("\nCreating SHAP Force Plots for individual samples...")
        sample_indices = [0, min(1, len(X_features_avg_clean)-1), min(2, len(X_features_avg_clean)-1)]
        sample_indices = [i for i in sample_indices if i < len(X_features_avg_clean)]

        try:
            for idx in sample_indices:
                 plt.figure(figsize=(12, 3))
                 shap.force_plot(
                     expected_value,
                     shap_values[idx, :],
                     X_features_avg_clean[idx, :],
                     feature_names=feature_names,
                     matplotlib=True,
                     show=False
                 )
                 plt.title(f"SHAP Force Plot for Sample {idx+1}")
                 plt.tight_layout()
                 plt.savefig(f"{shap_output_dir}/3_force_plot_sample_{idx+1}.png", dpi=300)
                 plt.close()
        except Exception as e:
             print(f"Error creating force plots: {e}")
             print("Creating alternative visualization for force plots...")

             # Create a custom horizontal bar chart as an alternative
             for idx in sample_indices:
                 plt.figure(figsize=(10, 6))
                 sample_shap = shap_values[idx, :]
                 sorted_idx = np.argsort(np.abs(sample_shap))
                 colors = ['red' if x < 0 else 'blue' for x in sample_shap[sorted_idx]]
                 plt.barh(
                     [feature_names[i] for i in sorted_idx],
                     sample_shap[sorted_idx],
                     color=colors
                 )
                 plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
                 plt.text(0, -1, f'Base value: {expected_value:.3f}', fontsize=12, ha='center', va='center')
                 final_pred = expected_value + np.sum(sample_shap)
                 plt.figtext(0.75, 0.01, f'Final prediction: {final_pred:.3f}', fontsize=12, ha='center')
                 plt.title(f"Feature Contributions for Sample {idx+1}")
                 plt.xlabel("SHAP Value (impact on prediction)")
                 plt.tight_layout()
                 plt.savefig(f"{shap_output_dir}/3_custom_force_plot_sample_{idx+1}.png", dpi=300)
                 plt.close()


        # 4. Alternative to Stacked Force Plots (Heatmap)
        print("\nCreating Alternative to Stacked Force Plots (Heatmap)...")
        num_stack_samples = min(20, len(X_features_avg_clean))
        if num_stack_samples > 0:
            stack_indices = range(num_stack_samples)
            plt.figure(figsize=(14, 10))
            shap_data = shap_values[stack_indices, :]
            abs_max = np.max(np.abs(shap_data))
            if abs_max == 0: abs_max = 1
            colors = ['#ff0000', '#ffffff', '#0000ff']
            cmap = LinearSegmentedColormap.from_list('custom_diverging', colors, N=256)
            sns.heatmap(
                 shap_data,
                 cmap=cmap,
                 vmin=-abs_max,
                 vmax=abs_max,
                 yticklabels=[f"Sample {i+1}" for i in stack_indices],
                 xticklabels=feature_names,
                 center=0,
                 annot=True,
                 fmt='.2f',
                 linewidths=0.5
            )
            plt.title(f"SHAP Values Heatmap for {num_stack_samples} Samples", fontsize=16)
            plt.tight_layout()
            plt.savefig(f"{shap_output_dir}/4_shap_values_heatmap.png", dpi=300)
            plt.close()
        else:
            print("Not enough samples for SHAP values heatmap.")


        # 5. Decision Plot
        print("\nCreating SHAP Decision Plots...")
        try:
            if len(X_features_avg_clean) > 0:
                 plt.figure(figsize=(12, 8))
                 shap.decision_plot(
                     expected_value,
                     shap_values[0:min(5, len(shap_values))],
                     X_features_avg_clean[0:min(5, len(X_features_avg_clean))],
                     feature_names=feature_names,
                     show=False
                 )
                 plt.title(f"SHAP Decision Plot")
                 plt.tight_layout()
                 plt.savefig(f"{shap_output_dir}/5_decision_plot.png", dpi=300)
                 plt.close()
            else:
                print("Not enough samples for Decision plots.")
        except Exception as e:
            print(f"Error creating decision plots: {e}")
            print("Creating alternative visualization for decision plots...")
            for i in range(min(2, len(X_features_avg_clean))):
                 plt.figure(figsize=(10, 6))
                 idx = np.argsort(np.abs(shap_values[i]))
                 plt.barh(
                     [feature_names[j] for j in idx],
                     shap_values[i, idx],
                     color=['red' if x < 0 else 'blue' for x in shap_values[i, idx]]
                 )
                 plt.axvline(x=0, color='k', linestyle='-', alpha=0.3)
                 plt.title(f"Feature Contributions for Sample {i+1}")
                 plt.xlabel("SHAP Value (Impact on Prediction)")
                 plt.tight_layout()
                 plt.savefig(f"{shap_output_dir}/5_feature_contributions_sample_{i+1}.png", dpi=300)
                 plt.close()


        # 6. Waterfall Plot
        print("\nCreating SHAP Waterfall Plots...")
        try:
             for idx in sample_indices:
                 plt.figure(figsize=(12, 8))
                 shap.plots.waterfall(
                     shap.Explanation(
                         values=shap_values[idx],
                         base_values=expected_value,
                         data=X_features_avg_clean[idx],
                         feature_names=feature_names
                     ),
                     show=False
                 )
                 plt.title(f"SHAP Waterfall Plot for Sample {idx+1}")
                 plt.tight_layout()
                 plt.savefig(f"{shap_output_dir}/6_waterfall_plot_sample_{idx+1}.png", dpi=300)
                 plt.close()
        except Exception as e:
             print(f"Error creating waterfall plots: {e}")
             print("Creating alternative visualization for waterfall plots...")
             for idx in sample_indices:
                 plt.figure(figsize=(12, 8))
                 sorted_idx = np.argsort(np.abs(shap_values[idx]))[::-1]
                 base = expected_value
                 cumulative = [base]
                 for i in sorted_idx:
                     cumulative.append(cumulative[-1] + shap_values[idx, i])
                 x_names = ['Base value'] + [feature_names[i] for i in sorted_idx] + ['Prediction']
                 plt.plot(range(len(cumulative)), cumulative, 'k-', alpha=0.5)
                 plt.scatter(range(len(cumulative)), cumulative, s=50, color='blue')
                 for i in range(len(cumulative)-1):
                     if i == 0:
                         plt.annotate(f"Base: {cumulative[i]:.3f}", (i, cumulative[i]), textcoords="offset points", xytext=(0,10), ha='center')
                     else:
                         contribution = shap_values[idx, sorted_idx[i-1]]
                         plt.annotate(f"{contribution:.3f}", (i, cumulative[i]), textcoords="offset points", xytext=(0,10), ha='center')
                 final_pred_val = expected_value + np.sum(shap_values[idx])
                 plt.annotate(f"Prediction: {final_pred_val:.3f}", (len(cumulative)-1, cumulative[-1]), textcoords="offset points", xytext=(0,10), ha='center', va='bottom', fontweight='bold')
                 plt.xticks(range(len(x_names)), x_names, rotation=90)
                 plt.title(f"Custom Waterfall Plot for Sample {idx+1}")
                 plt.ylabel("Prediction Value")
                 plt.grid(True, linestyle='--', alpha=0.3)
                 plt.tight_layout()
                 plt.savefig(f"{shap_output_dir}/6_custom_waterfall_plot_sample_{idx+1}.png", dpi=300)
                 plt.close()


        # 7. Dependence Scatter Plots showing effect of a single feature across dataset
        print("\nCreating SHAP Dependence Scatter Plots...")
        top_feature_indices = feature_ranks[:min(7, n_features)]

        for feature_idx in top_feature_indices:
             feature_name = feature_names[feature_idx]
             plt.figure(figsize=(10, 6))
             feature_values = X_features_avg_clean[:, feature_idx]
             feature_shap_values = shap_values[:, feature_idx]
             plt.scatter(feature_values, feature_shap_values, alpha=0.6, s=50)
             plt.xlabel(f"{feature_name} Value")
             plt.ylabel(f"SHAP Value (impact on prediction)")
             plt.title(f"SHAP Dependence Scatter Plot for {feature_name}")
             try:
                 if len(np.unique(feature_values)) > 1:
                     z = np.polyfit(feature_values, feature_shap_values, 1)
                     p = np.poly1d(z)
                     plt.plot(np.sort(feature_values), p(np.sort(feature_values)), "r--", linewidth=2)
                 else:
                     print(f"Skipping trend line for {feature_name}: Feature values are constant.")
             except Exception as e:
                 print(f"Warning: Could not plot trend line for {feature_name}: {e}")
             plt.grid(True, linestyle='--', alpha=0.7)
             plt.tight_layout()
             plt.savefig(f"{shap_output_dir}/7_dependence_scatter_{feature_name.lower().replace(' ', '_')}.png", dpi=300)
             plt.close()


        # 8. Beeswarm Plot
        print("\nCreating SHAP Beeswarm Plot...")
        try:
             if len(X_features_avg_clean) > 0:
                 plt.figure(figsize=(12, 8))
                 shap.plots.beeswarm(
                     shap.Explanation(
                         values=shap_values,
                         base_values=np.repeat(expected_value, shap_values.shape[0]),
                         data=X_features_avg_clean,
                         feature_names=feature_names
                     ),
                     show=False
                 )
                 plt.title(f"SHAP Beeswarm Plot: {model_name}")
                 plt.tight_layout()
                 plt.savefig(f"{shap_output_dir}/8_beeswarm_plot.png", dpi=300)
                 plt.close()
             else:
                 print("Not enough samples for Beeswarm plot.")
        except Exception as e:
             print(f"Error creating beeswarm plot: {e}")
             print("Creating alternative visualization for beeswarm...")
             plt.figure(figsize=(12, 8))
             for i, feature in enumerate(feature_names):
                 feature_shap = shap_values[:, i]
                 x = np.random.normal(i, 0.1, size=len(feature_shap))
                 plt.scatter(
                     x, feature_shap,
                     alpha=0.5,
                     c=feature_shap,
                     cmap='coolwarm',
                     s=20
                 )
             plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
             plt.xticks(range(len(feature_names)), feature_names, rotation=45, ha='right')
             plt.colorbar(label='SHAP Value')
             plt.title(f"Custom SHAP Distribution Plot: {model_name}")
             plt.ylabel("SHAP Value (impact on model output)")
             plt.tight_layout()
             plt.savefig(f"{shap_output_dir}/8_custom_shap_distribution.png", dpi=300)
             plt.close()


        # 9. Bar Plot of Mean Absolute SHAP Values
        print("\nCreating SHAP Bar Plot of Mean Absolute Values...")
        plt.figure(figsize=(12, 8))
        sorted_idx = np.argsort(mean_abs_shap)
        plt.barh(
            [feature_names[i] for i in sorted_idx],
            mean_abs_shap[sorted_idx],
            color='royalblue'
        )
        plt.xlabel("Mean |SHAP Value|")
        plt.title(f"Feature Importance (Mean Absolute SHAP Values): {model_name}")
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(f"{shap_output_dir}/9_mean_abs_shap_bar_plot.png", dpi=300)
        plt.close()

        # 10. SHAP Interaction Values
        print("\nCalculating SHAP Interaction Values...")
        interaction_samples = min(100, len(X_features_avg_clean))
        if interaction_samples > 1:
            try:
                 interaction_explainer = shap.TreeExplainer(surrogate_model)
                 interaction_values = interaction_explainer.shap_interaction_values(X_features_avg_clean[:interaction_samples])
                 interaction_magnitude = np.abs(interaction_values).mean(axis=0)

                 # Interaction with the most important feature
                 top_feature_idx = feature_ranks[0]
                 top_feature_name = feature_names[top_feature_idx]
                 interactions_with_top = interaction_magnitude[top_feature_idx, :]
                 other_feature_indices = [i for i in range(n_features) if i != top_feature_idx]
                 interacting_feature_names = [feature_names[i] for i in other_feature_indices]
                 interacting_feature_values = interactions_with_top[other_feature_indices]
                 sort_order = np.argsort(interacting_feature_values)[::-1]

                 plt.figure(figsize=(12, 10))
                 plt.bar(
                     [interacting_feature_names[i] for i in sort_order],
                     [interacting_feature_values[i] for i in sort_order],
                     color='royalblue'
                 )
                 plt.xlabel(f"Feature interacting with {top_feature_name}")
                 plt.ylabel("Interaction Strength (Mean |SHAP Interaction Value|)")
                 plt.title(f"SHAP Interaction Values with {top_feature_name}")
                 plt.xticks(rotation=45, ha='right')
                 plt.grid(True, linestyle='--', alpha=0.7)
                 plt.tight_layout()
                 plt.savefig(f"{shap_output_dir}/10_interaction_values_{top_feature_name.lower().replace(' ', '_')}.png", dpi=300)
                 plt.close()

                 # Heatmap of all interaction values
                 plt.figure(figsize=(12, 10))
                 sns.heatmap(
                     interaction_magnitude,
                     xticklabels=feature_names,
                     yticklabels=feature_names,
                     cmap='YlOrRd',
                     annot=True,
                     fmt='.2f'
                 )
                 plt.title("SHAP Feature Interaction Heatmap (Mean Absolute Values)")
                 plt.tight_layout()
                 plt.savefig(f"{shap_output_dir}/10_interaction_heatmap.png", dpi=300)
                 plt.close()

            except Exception as e:
                 print(f"Error calculating interaction values: {e}")
                 print("Skipping SHAP interaction analysis.")
        else:
            print("Not enough samples to calculate SHAP interaction values. Skipping interaction analysis.")


        # --- WMA Mapping Section ---
        print("\n--- Mapping SHAP values to WMAs ---")

        if ids_sample is not None:
            try:
                # Aggregate SHAP values by WMA ID using ids_sample which matches shap_values
                # Create a DataFrame from shap_values with ids_sample as index
                wma_shap_aggregated = pd.DataFrame(shap_values, index=ids_sample)
                wma_shap_aggregated.index.name = 'WMA_ID_Data'

                # Calculate mean SHAP values for each unique WMA ID in the sample
                # Use reset_index() to make WMA_ID_Data a column for merging
                wma_mean_shap_df = wma_shap_aggregated.groupby('WMA_ID_Data').mean().reset_index()
                wma_mean_shap_df.columns = ['WMA_ID_Data'] + [f'{feat}_SHAP_Mean' for feat in feature_names] # Rename columns

                print(f"Calculated mean SHAP values for {len(wma_mean_shap_df)} unique WMAs in the sample data.")

                # Load WMA shapefile
                print(f"Loading WMA shapefile from {wma_shapefile_path}...")
                try:
                    # Check if file exists, try alternative paths if not found
                    current_shapefile_path = wma_shapefile_path
                    if not os.path.exists(current_shapefile_path):
                        print(f"Warning: Specified shapefile not found at {current_shapefile_path}")
                        print("Checking alternative paths...")
                        potential_shapefile_paths = [
                            f"./{wma_shapefile_path}",
                            f"/{wma_shapefile_path}",
                            "/glade/work/yifanl/ConvLSTM/simpleWMAs_v3_ID.shp",
                            "./simpleWMAs_v3_ID.shp",
                            "simpleWMAs_v3_ID.shp"
                        ]
                        found_path = None
                        for path in potential_shapefile_paths:
                            if os.path.exists(path):
                                found_path = path
                                print(f"Found shapefile at {path}")
                                break
                        if found_path:
                            current_shapefile_path = found_path
                        else:
                            raise FileNotFoundError(f"Could not find WMA shapefile at {wma_shapefile_path} or any alternative paths")

                    wma_gdf = gpd.read_file(current_shapefile_path)
                    print(f"Loaded shapefile with {len(wma_gdf)} WMA boundaries.")

                    # Identify the correct ID field in the shapefile for merging
                    id_field_to_use = None
                    possible_id_fields = ['ID', 'WMA_ID', 'WMAID', 'OBJECTID', 'FID', 'id']
                    for field in possible_id_fields:
                         if field in wma_gdf.columns:
                             id_field_to_use = field
                             print(f"Using '{field}' as the ID field from the shapefile for merging.")
                             break

                    if id_field_to_use is None:
                        raise ValueError("Could not find a suitable ID field in the shapefile columns for merging.")

                    print(f"Shapefile preview ({id_field_to_use} column and first 5 rows):")
                    preview_cols = [id_field_to_use] + [col for col in wma_gdf.columns if col != id_field_to_use][:4]
                    print(wma_gdf[preview_cols].head())

                    # Ensure ID types match for merging
                    try:
                        wma_gdf[id_field_to_use] = wma_gdf[id_field_to_use].astype(str)
                        wma_mean_shap_df['WMA_ID_Data'] = wma_mean_shap_df['WMA_ID_Data'].astype(str)
                        print("Converted ID columns to string for merging.")
                    except Exception as type_err:
                        print(f"Warning: Could not convert ID columns to string: {type_err}")


                    # Merge the GeoDataFrame with the calculated mean SHAP values DataFrame
                    wma_gdf_merged = wma_gdf.merge(
                        wma_mean_shap_df,
                        left_on=id_field_to_use,
                        right_on='WMA_ID_Data',
                        how='left'
                    )

                    # Drop the redundant ID column from the SHAP data after merging
                    wma_gdf_merged = wma_gdf_merged.drop(columns=['WMA_ID_Data'])

                    # Check how many WMAs were matched
                    shap_mean_cols = [f'{feat}_SHAP_Mean' for feat in feature_names] # Redefine for merged data
                    matched_wmas = wma_gdf_merged.dropna(subset=[shap_mean_cols[0]])
                    print(f"Successfully matched SHAP data to {len(matched_wmas)} WMA boundaries out of {len(wma_gdf)} in the shapefile.")
                    if len(matched_wmas) < len(wma_mean_shap_df):
                        unmatched_data_ids = set(wma_mean_shap_df['WMA_ID_Data']) - set(wma_gdf_merged[id_field_to_use].dropna())
                        print(f"Warning: {len(unmatched_data_ids)} WMA IDs from SHAP data were not found in the shapefile's '{id_field_to_use}' column and will not appear on the map.")


                    # Calculate overall importance based on the merged SHAP values
                    wma_gdf_merged['OVERALL_IMPORTANCE'] = wma_gdf_merged[shap_mean_cols].abs().sum(axis=1)

                except Exception as shape_err:
                     print(f"Error during WMA shapefile loading or merging: {shape_err}")
                     import traceback
                     traceback.print_exc()
                     print("Skipping WMA mapping visualization due to error.")
                     wma_gdf_merged = None # Set to None if merge fails completely


                # --- Create WMA Choropleth Maps ---
                if wma_gdf_merged is not None:
                    print("\n--- Generating WMA Maps ---")

                    # 11a. Individual Choropleth Maps for each feature
                    for feature_idx, feature_name in enumerate(feature_names):
                         shap_col_name = shap_mean_cols[feature_idx]
                         print(f"Creating individual choropleth map for {feature_name}...")

                         if wma_gdf_merged[shap_col_name].dropna().empty:
                             print(f"No SHAP data available for {feature_name} after merging. Skipping map.")
                             continue

                         fig, ax = plt.subplots(1, 1, figsize=(12, 8))
                         vmin = wma_gdf_merged[shap_col_name].min()
                         vmax = wma_gdf_merged[shap_col_name].max()
                         abs_max = max(abs(vmin), abs(vmax))
                         if abs_max == 0:
                             print(f"SHAP values for {feature_name} are all zero. Skipping map.")
                             plt.close(fig)
                             continue

                         wma_gdf_merged.plot(
                             column=shap_col_name,
                             ax=ax,
                             cmap='coolwarm',
                             vmin=-abs_max,
                             vmax=abs_max,
                             legend=True,
                             edgecolor='black',
                             linewidth=0.5,
                             missing_kwds={"color": "lightgrey", "edgecolor": "red", "hatch": "///", "label": "Missing data"},
                             legend_kwds={'label': f'Mean SHAP Value for {feature_name}', 'orientation': "horizontal", 'shrink': 0.8}
                         )
                         plt.title(f"Mean SHAP Values for {feature_name} Across WMAs")
                         plt.axis('off')
                         plt.tight_layout()
                         plt.savefig(f"{wma_output_dir}/11a_wma_map_{feature_name.lower().replace(' ', '_')}.png", dpi=300)
                         plt.close()


                    # 11b. Combined Choropleth Map for All Features
                    print("\nCreating combined choropleth map for all features...")
                    n_cols = int(np.ceil(np.sqrt(n_features)))
                    n_rows = int(np.ceil(n_features / n_cols))
                    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
                    axes = axes.flatten()

                    all_mean_shap_values = wma_gdf_merged[shap_mean_cols].values.flatten()
                    global_max_abs_shap = np.nanmax(np.abs(all_mean_shap_values))
                    if np.isnan(global_max_abs_shap) or global_max_abs_shap == 0:
                        print("No non-zero SHAP data available for combined map. Skipping.")
                        plt.close(fig)
                    else:
                        sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=-global_max_abs_shap, vmax=global_max_abs_shap))
                        sm._A = []

                        for feature_idx, feature_name in enumerate(feature_names):
                            shap_col_name = shap_mean_cols[feature_idx]
                            ax = axes[feature_idx]
                            wma_gdf_merged.plot(
                                column=shap_col_name,
                                ax=ax,
                                cmap='coolwarm',
                                vmin=-global_max_abs_shap,
                                vmax=global_max_abs_shap,
                                edgecolor='black',
                                linewidth=0.3,
                                missing_kwds={"color": "lightgrey", "hatch": "///"},
                            )
                            ax.set_title(feature_name, fontsize=10)
                            ax.axis('off')

                        for i in range(n_features, len(axes)):
                            fig.delaxes(axes[i])

                        cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.03])
                        cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal", label="Mean SHAP Value")
                        cbar.ax.tick_params(labelsize=8)

                        plt.suptitle("Mean SHAP Values Across WMAs by Feature", fontsize=16, y=0.98)
                        plt.tight_layout(rect=[0, 0.08, 1, 0.95])
                        plt.savefig(f"{wma_output_dir}/11b_wma_map_all_features_combined.png", dpi=300)
                        plt.close()


                    # 11c. Overall Importance Map
                    print("Creating overall importance map...")
                    if wma_gdf_merged['OVERALL_IMPORTANCE'].dropna().empty:
                         print("No overall importance data available. Skipping map.")
                    else:
                         fig, ax = plt.subplots(1, 1, figsize=(12, 8))
                         wma_gdf_merged.plot(
                             column='OVERALL_IMPORTANCE',
                             ax=ax,
                             cmap='viridis',
                             legend=True,
                             edgecolor='black',
                             linewidth=0.5,
                             missing_kwds={"color": "lightgrey", "edgecolor": "red", "hatch": "///", "label": "Missing data"},
                             legend_kwds={'label': 'Overall Mean SHAP Importance', 'orientation': "horizontal", 'shrink': 0.8}
                         )
                         plt.title(f"Overall Mean SHAP Importance Across WMAs")
                         plt.axis('off')
                         plt.tight_layout()
                         plt.savefig(f"{wma_output_dir}/11c_wma_map_overall_importance.png", dpi=300)
                         plt.close()

                    # Save the WMA SHAP data to a CSV (using the merged dataframe)
                    cols_to_save = [id_field_to_use, 'OVERALL_IMPORTANCE'] + shap_mean_cols
                    wma_df_to_csv = wma_gdf_merged[cols_to_save].copy()
                    wma_df_to_csv = wma_df_to_csv.rename(columns={id_field_to_use: 'WMA_ID_Shapefile'})
                    wma_df_to_csv = wma_df_to_csv.sort_values('OVERALL_IMPORTANCE', ascending=False)
                    wma_df_to_csv.to_csv(f"{wma_output_dir}/wma_mean_shap_values.csv", index=False)
                    print(f"Mean SHAP values per WMA saved to '{wma_output_dir}/wma_mean_shap_values.csv'")


                    # Perform detailed analysis on top 5 WMAs with highest overall importance
                    print("\nPerforming detailed analysis on top 5 WMAs with highest overall importance...")
                    wma_gdf_with_data = wma_gdf_merged.dropna(subset=['OVERALL_IMPORTANCE']).copy() # Use copy to avoid SettingWithCopyWarning

                    if not wma_gdf_with_data.empty:
                         top5_wmas_gdf = wma_gdf_with_data.nlargest(5, 'OVERALL_IMPORTANCE')
                         top5_wma_ids_shapefile = top5_wmas_gdf[id_field_to_use].tolist()
                         top5_importance = top5_wmas_gdf['OVERALL_IMPORTANCE'].tolist()

                         print(f"Top {len(top5_wmas_gdf)} WMAs by overall importance (Shapefile ID):")
                         for i, (wma_id_shapefile, importance) in enumerate(zip(top5_wma_ids_shapefile, top5_importance)):
                              print(f"{i+1}. WMA Shapefile ID {wma_id_shapefile}: {importance:.4f}")

                         # Create detailed visualization of feature importance for top 5 WMAs
                         top5_feature_values_mean = top5_wmas_gdf[shap_mean_cols].values

                         if len(top5_wmas_gdf) > 0:
                              plt.figure(figsize=(12, len(top5_wmas_gdf) * 2))
                              for i in range(len(top5_wmas_gdf)):
                                  wma_id_shapefile = top5_wmas_ids_shapefile[i]
                                  importance = top5_importance[i]
                                  feature_values = top5_feature_values_mean[i]
                                  plt.subplot(len(top5_wmas_gdf), 1, i+1)
                                  colors = ['red' if v < 0 else 'blue' for v in feature_values]
                                  sort_order = np.argsort(feature_values)
                                  plt.barh([feature_names[j] for j in sort_order], feature_values[sort_order], color=[colors[j] for j in sort_order])
                                  plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
                                  plt.title(f"WMA Shapefile ID {wma_id_shapefile} (Importance: {importance:.4f})", fontsize=10)
                                  if i < len(top5_wmas_gdf) - 1:
                                      plt.xticks([])
                                  plt.xlabel("Mean SHAP Value" if i == len(top5_wmas_gdf) -1 else "")
                                  plt.grid(axis='x', linestyle='--', alpha=0.7)
                                  plt.tight_layout()

                              plt.suptitle("Mean Feature Importance Breakdown for Top WMAs", fontsize=16, y=1.02)
                              plt.tight_layout(rect=[0, 0, 1, 0.98])
                              plt.savefig(f"{wma_output_dir}/top_wmas/feature_breakdown.png", dpi=300, bbox_inches='tight')
                              plt.close()

                              # Heatmap showing all features for top WMAs
                              plt.figure(figsize=(12, min(len(top5_wmas_gdf), 10) * 0.7 + 1))
                              top5_heatmap_df = pd.DataFrame(
                                  top5_feature_values_mean,
                                  columns=feature_names,
                                  index=[f"WMA {wma_id}" for wma_id in top5_wma_ids_shapefile]
                              )
                              max_abs = np.nanmax(np.abs(top5_feature_values_mean))
                              if not np.isnan(max_abs) and max_abs > 0:
                                  sns.heatmap(
                                      top5_heatmap_df,
                                      cmap='coolwarm',
                                      annot=True,
                                      fmt='.3f',
                                      center=0,
                                      vmin=-max_abs,
                                      vmax=max_abs
                                  )
                                  plt.title("Mean SHAP Values by Feature for Top WMAs")
                                  plt.tight_layout()
                                  plt.savefig(f"{wma_output_dir}/top_wmas/feature_heatmap.png", dpi=300)
                                  plt.close()

                              # Map highlighting the top WMAs
                              print("Creating zoomed maps highlighting top WMAs...")
                              fig, ax = plt.subplots(figsize=(12, 8))
                              wma_gdf_merged.plot(
                                  ax=ax, color='lightgray', edgecolor='darkgray', linewidth=0.5,
                                  missing_kwds={"color": "lightgrey", "hatch": "///"}
                              )
                              colors = plt.cm.tab10(np.linspace(0, 1, len(top5_wmas_gdf)))
                              for i, (idx, row) in enumerate(top5_wmas_gdf.iterrows()):
                                   wma_id_shapefile = row[id_field_to_use]
                                   highlight_gdf = wma_gdf_merged[wma_gdf_merged[id_field_to_use] == wma_id_shapefile]
                                   if not highlight_gdf.empty:
                                       highlight_gdf.plot(
                                           ax=ax, color=colors[i], edgecolor='black', linewidth=1
                                       )
                                       if highlight_gdf.geometry.iloc[0] and not highlight_gdf.geometry.iloc[0].is_empty:
                                            centroid = highlight_gdf.geometry.iloc[0].centroid
                                            ax.annotate(
                                                f"{wma_id_shapefile}", (centroid.x, centroid.y),
                                                fontsize=10, fontweight='bold', ha='center', va='center',
                                                bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', boxstyle='round,pad=0.3')
                                            )

                              plt.title("Top WMAs by Overall Mean SHAP Importance (Shapefile ID)")
                              plt.axis('off')
                              plt.tight_layout()
                              plt.savefig(f"{wma_output_dir}/top_wmas/top_highlighted.png", dpi=300)
                              plt.close()

                              # Individual zoomed maps for each top WMA
                              for i, (idx, row) in enumerate(top5_wmas_gdf.iterrows()):
                                   wma_id_shapefile = row[id_field_to_use]
                                   overall_importance = row['OVERALL_IMPORTANCE']
                                   feature_values = row[shap_mean_cols].values
                                   highlight_gdf = wma_gdf_merged[wma_gdf_merged[id_field_to_use] == wma_id_shapefile]
                                   if not highlight_gdf.empty:
                                       fig, ax = plt.subplots(1, 1, figsize=(10, 10))
                                       minx, miny, maxx, maxy = highlight_gdf.geometry.total_bounds
                                       buffer = max((maxx - minx), (maxy - miny)) * 0.5 * 0.2
                                       ax.set_xlim(minx - buffer, maxx + buffer)
                                       ax.set_ylim(miny - buffer, maxy + buffer)

                                       wma_gdf_merged.plot(
                                            ax=ax, color='lightgray', edgecolor='darkgray', linewidth=0.5,
                                            missing_kwds={"color": "lightgrey", "hatch": "///"},
                                       )
                                       highlight_gdf.plot(
                                            ax=ax, color=colors[i % len(colors)], edgecolor='black', linewidth=1.5
                                       )
                                       if highlight_gdf.geometry.iloc[0] and not highlight_gdf.geometry.iloc[0].is_empty:
                                           centroid = highlight_gdf.geometry.iloc[0].centroid
                                           ax.annotate(
                                                f"WMA {wma_id_shapefile}", (centroid.x, centroid.y),
                                                fontsize=12, fontweight='bold', ha='center', va='center',
                                                bbox=dict(facecolor='white', alpha=0.8, edgecolor='black', boxstyle='round,pad=0.3')
                                           )

                                       inset_ax = fig.add_axes([0.6, 0.05, 0.35, 0.3])
                                       colors_bars = ['red' if v < 0 else 'blue' for v in feature_values]
                                       sort_order = np.argsort(np.abs(feature_values))[::-1]
                                       inset_ax.barh(
                                           [feature_names[j] for j in sort_order],
                                           [feature_values[j] for j in sort_order],
                                           color=[colors_bars[j] for j in sort_order]
                                       )
                                       inset_ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
                                       inset_ax.set_title("Mean Feature Importance", fontsize=10)
                                       inset_ax.tick_params(axis='y', labelsize=8)
                                       inset_ax.tick_params(axis='x', labelsize=8)

                                       plt.suptitle(f"WMA Shapefile ID {wma_id_shapefile} - Overall Importance: {overall_importance:.4f}", fontsize=14, y=0.95)
                                       plt.axis('off')
                                       plt.savefig(f"{wma_output_dir}/top_wmas/wma_{wma_id_shapefile}_zoom.png", dpi=300, bbox_inches='tight')
                                       plt.close()

                              # Detailed statistics table for top WMAs
                              top_stats_df = top5_wmas_gdf[[id_field_to_use, 'OVERALL_IMPORTANCE'] + shap_mean_cols].copy()
                              top_stats_df = top_stats_df.rename(columns={id_field_to_use: 'WMA_ID_Shapefile'})
                              top_stats_df['Dominant_Positive_Feature'] = np.argmax(top5_feature_values_mean, axis=1)
                              top_stats_df['Dominant_Positive_Feature'] = top_stats_df['Dominant_Positive_Feature'].apply(lambda x: feature_names[x])
                              top_stats_df['Positive_Impact'] = np.max(top5_feature_values_mean, axis=1)
                              top_stats_df['Dominant_Negative_Feature'] = np.argmin(top5_feature_values_mean, axis=1)
                              top_stats_df['Dominant_Negative_Feature'] = top_stats_df['Dominant_Negative_Feature'].apply(lambda x: feature_names[x])
                              top_stats_df['Negative_Impact'] = np.min(top5_feature_values_mean, axis=1)
                              top_stats_df.to_csv(f"{wma_output_dir}/top_wmas/top_wma_statistics.csv", index=False)
                              print(f"Detailed analysis of top {len(top5_wmas_gdf)} WMAs completed. Results saved to '{wma_output_dir}/top_wmas/'")

                         else:
                              print("No WMAs with SHAP data found in the shapefile for top WMA analysis.")

                    else:
                        print("WMA mapping visualization skipped due to errors during merging.")

            except Exception as e:
                print(f"Error in WMA mapping section: {e}")
                import traceback
                traceback.print_exc()
                print("WMA mapping failed.")
        else:
            print("No WMA IDs provided in ids_test. Skipping WMA mapping.")


        print("\nAll analysis steps completed.")
        return True

    except Exception as e:
        print(f"An unexpected error occurred during analysis: {e}")
        import traceback
        traceback.print_exc()
        print("Analysis failed.")
        return False

# ===== MAIN EXECUTION =====
if __name__ == "__main__":
    print("\n===== STARTING INTEGRATED SHAP AND WMA MAPPING ANALYSIS =====")
    print(f"Model: {model_name}")
    print(f"Data shape: {X_test.shape}")

    # Run the combined analysis function
    success = run_analysis(
        model,
        X_test,
        y_test,
        ids_test=ids_test,
        max_samples_shap=20000
    )

    if success:
        print("\n===== ANALYSIS COMPLETE =====")
        print(f"All visualizations and data files have been saved to '{output_base_dir}'.")
    else:
        print("\n===== ANALYSIS FAILED =====")
        print("Check the error messages above for details.")
        print("Ensure all required libraries are installed, data is correctly formatted, and shapefile path and ID field are correct.")


===== Loading Model and Data =====
Loading ConvLSTM2D ReLU from water_demand_convlstm2d_relu_model.keras...


2025-05-05 23:38:31.058281: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


ConvLSTM2D ReLU loaded successfully.
Loading analysis data from analysis_outputs/water_demand_model_data.pkl...
Data loaded successfully:
  - X_test shape: (18447, 12, 27, 27, 7)
  - y_test shape: (18447,)
  - ids_test shape: (18447,)

===== STARTING INTEGRATED SHAP AND WMA MAPPING ANALYSIS =====
Model: ConvLSTM2D ReLU
Data shape: (18447, 12, 27, 27, 7)

===== RUNNING ANALYSIS (SHAP + WMA Mapping) =====

--- Calculating SHAP Values ---
SHAP analysis on 18447 samples...
Training surrogate model for SHAP analysis...
Creating SHAP explainer...
Calculating SHAP values...

SHAP Feature Importance Results:
                  Feature  Importance  Mean_Abs_SHAP
1           Precipitation    0.228909       0.083902
5      Crop Area Fraction    0.217343       0.079663
3    Domestic Water Usage    0.155145       0.056865
0             Temperature    0.136842       0.050157
4  Industrial Water Usage    0.121548       0.044551
6   Irrigation Efficiency    0.095516       0.035009
2      Population Den

/glade/derecho/scratch/yifanl/tmp/ipykernel_101409/3112372351.py:730: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.08, 1, 0.95])


Creating overall importance map...
Mean SHAP values per WMA saved to 'analysis_outputs/wma_maps/wma_mean_shap_values.csv'

Performing detailed analysis on top 5 WMAs with highest overall importance...
Top 5 WMAs by overall importance (Shapefile ID):
1. WMA Shapefile ID 378: 0.7142
2. WMA Shapefile ID 584: 0.7135
3. WMA Shapefile ID 588: 0.7094
4. WMA Shapefile ID 167: 0.7038
5. WMA Shapefile ID 287: 0.7035
Error in WMA mapping section: name 'top5_wmas_ids_shapefile' is not defined
WMA mapping failed.

All analysis steps completed.

===== ANALYSIS COMPLETE =====
All visualizations and data files have been saved to 'analysis_outputs'.


...

<Figure size 1000x600 with 0 Axes>

<Figure size 1200x300 with 0 Axes>

<Figure size 1200x300 with 0 Axes>

<Figure size 1200x300 with 0 Axes>

<Figure size 1200x1000 with 0 Axes>